# PhishGuard — Eksplorasi Data & Analisis Karakteristik URL

Notebook ini bertujuan untuk menganalisis karakteristik distribusi fitur pada dataset **PhishGuard** (`data/processed/train.csv`, `val.csv`, `test.csv`).

## Tujuan Analisis:
1. Memeriksa keseimbangan kelas dan distribusi partisi (train/val/test).
2. Menganalisis perbedaan panjang URL dan panjang host antara legitimate vs phishing.
3. Memeriksa distribusi TLD (Top-Level Domains) yang sering digunakan pada phishing.
4. Mengamati frekuensi karakter khusus (titik, tanda hubung, simbol, digit).
5. Menemukan wawasan kunci untuk tahap ekstraksi fitur (Milestone M03).

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tldextract

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)

# Load dataset splits
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print(f"Train samples: {len(train_df):,} | Val samples: {len(val_df):,} | Test samples: {len(test_df):,}")

## 1. Distribusi Kelas pada Setiap Split
Periksa rasio phishing (1) vs legitimate (0) untuk memastikan tidak terjadi ketidakseimbangan ekstrem.

In [ ]:
split_summary = []
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    counts = df["label"].value_counts()
    split_summary.append({
        "Split": name,
        "Total": len(df),
        "Legitimate (0)": counts.get(0, 0),
        "Phishing (1)": counts.get(1, 0),
        "Phishing %": f"{df['label'].mean() * 100:.2f}%",
        "Unique Domains": df["domain"].nunique()
    })

pd.DataFrame(split_summary)

## 2. Distribusi Panjang URL (URL Length)
URL phishing umumnya lebih panjang karena menyematkan token acak, redirect, atau subdomain bertumpuk.

In [ ]:
train_df["url_len"] = train_df["url"].str.len()
print("Statistik deskriptif panjang URL:")
print(train_df.groupby("label")["url_len"].describe())

## 3. Distribusi Top-Level Domains (TLD)
Analisis ekstensi domain (TLD) yang paling sering muncul pada URL phishing dibandingkan legitimate.

In [ ]:
def get_tld(domain_str):
    if not isinstance(domain_str, str) or "." not in domain_str:
        return "ip_or_none"
    return domain_str.split(".")[-1]

train_df["tld"] = train_df["domain"].apply(get_tld)

top_phish_tlds = train_df[train_df["label"] == 1]["tld"].value_counts().head(10)
top_legit_tlds = train_df[train_df["label"] == 0]["tld"].value_counts().head(10)

tld_comparison = pd.DataFrame({
    "Top Phishing TLDs": top_phish_tlds.index,
    "Phishing Count": top_phish_tlds.values,
    "Top Legit TLDs": top_legit_tlds.index,
    "Legit Count": top_legit_tlds.values
})
tld_comparison

## 4. Karakter Khusus & Fitur Lexical Kunci
Bandingkan rata-rata kemunculan karakter khusus seperti titik (`.`), tanda hubung (`-`), `@`, `?`, `=`, serta rasio digit.

In [ ]:
lexical_features = pd.DataFrame({
    "avg_dots": train_df.groupby("label")["nb_dots"].mean(),
    "avg_hyphens": train_df.groupby("label")["nb_hyphens"].mean(),
    "avg_at": train_df.groupby("label")["nb_at"].mean(),
    "avg_qm": train_df.groupby("label")["nb_qm"].mean(),
    "avg_eq": train_df.groupby("label")["nb_eq"].mean(),
    "avg_ratio_digits": train_df.groupby("label")["ratio_digits_url"].mean()
})
lexical_features.T

## 5. Kesimpulan & Rekomendasi Fitur untuk M03
- **Panjang URL & Host:** URL phishing secara signifikan memiliki rata-rata panjang yang lebih besar.
- **Karakter Pemisah & Simbol:** Tanda hubung (`-`), tanda tanya (`?`), dan simbol sama dengan (`=`) jauh lebih lazim di phishing.
- **Rasio Digit & Entropi:** Pola string acak (hex hash, session id tiruan) pada phishing menghasilkan rasio angka dan entropi karakter yang lebih tinggi.
- **Indikasi Merek / Kata Kunci Sensitif:** Perlu kamus kata kunci sensitif (`login`, `secure`, `verify`, `account`, `bank`) serta kata lokal Indonesia (`akun`, `masuk`, `pembayaran`).